In [1]:
from __future__ import annotations

import copy
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset

In [2]:
# ============================================================
# CONFIG
# ============================================================

PREPROCESSED_ROOT = Path(r"D:\HUP_processed_ver2")
EXPERIMENT_ROOT = Path(r"D:\hup_all_subjects_ver2")
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

SUBJECT_FILTER = [
    "sub-HUP126",
    "sub-HUP164",
    "sub-HUP130",
    "sub-HUP157",
]

LLM_COHORT_SUBJECTS: List[str] = [
    "sub-HUP126",
    "sub-HUP164",
    "sub-HUP130",
    "sub-HUP157",
]

LLM_UNIFIED_ROOT_BASE = Path(r"D:\LLM_outputs_for_subjects")
LLM_UNIFIED_ROOT_BASE.mkdir(parents=True, exist_ok=True)
# Example:
# MANUAL_LLM_HOLDOUTS = {
#     "sub-HUP146": {
#         "ictal": "sub-HUP146_ses-presurgery_task-ictal_acq-seeg_run-03",
#         "interictal": "sub-HUP146_ses-presurgery_task-interictal_acq-seeg_run-02",
#     }
# }
MANUAL_LLM_HOLDOUTS: Dict[str, Dict[str, str]] = {}

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
NUM_EPOCHS = 35
LR = 3e-4
WEIGHT_DECAY = 3e-4
EARLY_STOPPING_PATIENCE = 5

DROPOUT = 0.30
CNN_HIDDEN = 32
EMBED_DIM = 64
NHEAD = 2
NUM_LAYERS = 1
FF_MULT = 4

FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.5

SMOOTHING_KERNEL = 5

# keep your new threshold grid unchanged
THRESH_GRID = [round(x, 2) for x in np.arange(0.10, 0.91, 0.05)]
FALLBACK_THRESHOLD = 0.50

# fixed per-subject retention for training only
MAX_TRAIN_ICTAL_PER_SUBJECT = 100
MAX_TRAIN_NONICTAL_PER_SUBJECT = 100

MIN_ICTAL_WINDOWS_FOR_TRAIN = 1

LLM_UNIFIED_ROOT = Path(r"D:\LLM_unified_HUP_ver2")


In [3]:
# ============================================================
# UTILS
# ============================================================

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)


def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1 or len(x) == 0:
        return x.copy()
    pad = k // 2
    xpad = np.pad(x, (pad, pad), mode="edge")
    kernel = np.ones(k, dtype=np.float32) / float(k)
    return np.convolve(xpad, kernel, mode="valid")


def safe_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_prob))


def safe_auprc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(average_precision_score(y_true, y_prob))


def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(np.int64)

    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    auroc = safe_roc_auc(y_true, y_prob)
    auprc = safe_auprc(y_true, y_prob)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "acc": float(acc),
        "balanced_acc": float(bal_acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1": float(f1),
        "auroc": float(auroc) if not math.isnan(auroc) else np.nan,
        "auprc": float(auprc) if not math.isnan(auprc) else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_best_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    thresh_grid: List[float],
    fallback_threshold: float = 0.5,
) -> float:
    if len(np.unique(y_true)) < 2:
        return fallback_threshold

    best_thr = fallback_threshold
    best_key = (-1.0, -1.0, -1.0)  # balanced_acc, f1, precision

    for thr in thresh_grid:
        m = compute_metrics(y_true, y_prob, thr)
        key = (m["balanced_acc"], m["f1"], m["precision"])
        if key > best_key:
            best_key = key
            best_thr = thr

    return float(best_thr)

In [4]:
# ============================================================
# DATA LOADING
# ============================================================

@dataclass
class RunRecord:
    subject_id: str
    run_stem: str
    npz_path: Path
    meta_path: Path
    task: str
    acquisition: str
    n_windows: int
    n_ictal: int
    n_nonictal: int


def parse_stem(stem: str) -> Dict[str, str]:
    # sub-HUP146_ses-presurgery_task-ictal_acq-seeg_run-03
    parts = stem.split("_")
    out = {
        "subject_id": parts[0],
        "session": parts[1],
        "task": parts[2].replace("task-", ""),
        "acquisition": parts[3].replace("acq-", ""),
        "run": parts[4].replace("run-", ""),
    }
    return out


def scan_runs(preprocessed_root: Path) -> List[RunRecord]:
    runs = []
    for npz_path in sorted(preprocessed_root.glob("sub-*/*.npz")):
        stem = npz_path.stem
        meta_path = npz_path.with_name(f"{stem}_meta.json")
        if not meta_path.exists():
            continue

        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)

        counts = meta.get("class_counts", {})
        runs.append(
            RunRecord(
                subject_id=meta["subject"],
                run_stem=stem,
                npz_path=npz_path,
                meta_path=meta_path,
                task=meta["task"],
                acquisition=meta["acquisition"],
                n_windows=int(counts.get("n_total", 0)),
                n_ictal=int(counts.get("n_ictal", 0)),
                n_nonictal=int(counts.get("n_nonictal", 0)),
            )
        )
    return sorted(runs, key=lambda r: (r.subject_id, r.task, r.run_stem))


def filter_runs(runs: List[RunRecord]) -> List[RunRecord]:
    out = []
    for r in runs:
        if SUBJECT_FILTER and r.subject_id not in SUBJECT_FILTER:
            continue
        # keep both ictal and interictal runs for LLM holdout logic
        if r.task not in {"ictal", "interictal"}:
            continue
        out.append(r)
    return out


def load_npz(npz_path: Path) -> Dict[str, np.ndarray]:
    d = np.load(npz_path)
    return {
        "X": d["X"].astype(np.float32),            # (N, C, T)
        "y": d["y"].astype(np.int64),              # eval labels
        "y_train": d["y_train"].astype(np.int64),  # -1 ignore, 0, 1
        "mask": d["mask"].astype(bool),            # (N, C)
        "t_bounds": d["t_bounds"].astype(np.float32),
        "rms_z": d["rms_z"].astype(np.float32),
        "peak_z": d["peak_z"].astype(np.float32),
    }


def concatenate_runs(run_paths: List[Path]) -> Dict[str, np.ndarray]:
    blocks = [load_npz(p) for p in run_paths]
    X = np.concatenate([b["X"] for b in blocks], axis=0)
    y = np.concatenate([b["y"] for b in blocks], axis=0)
    y_train = np.concatenate([b["y_train"] for b in blocks], axis=0)
    mask = np.concatenate([b["mask"] for b in blocks], axis=0)
    t_bounds = np.concatenate([b["t_bounds"] for b in blocks], axis=0)

    run_ids = []
    for b, p in zip(blocks, run_paths):
        run_ids.extend([p.stem] * len(b["y"]))
    run_ids = np.asarray(run_ids)

    return {
        "X": X,
        "y": y,
        "y_train": y_train,
        "mask": mask,
        "t_bounds": t_bounds,
        "run_ids": run_ids,
    }


def retain_subject_quota(
    X: np.ndarray,
    y_eval: np.ndarray,
    y_train: np.ndarray,
    mask: np.ndarray,
    t_bounds: np.ndarray,
    run_ids: np.ndarray,
    max_ictal: int = 100,
    max_nonictal: int = 100,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)

    valid_idx = np.where(y_train != -1)[0]
    ictal_idx = valid_idx[y_train[valid_idx] == 1]
    nonictal_idx = valid_idx[y_train[valid_idx] == 0]

    if len(ictal_idx) > max_ictal:
        ictal_idx = np.sort(rng.choice(ictal_idx, size=max_ictal, replace=False))
    if len(nonictal_idx) > max_nonictal:
        nonictal_idx = np.sort(rng.choice(nonictal_idx, size=max_nonictal, replace=False))

    keep_idx = np.sort(np.concatenate([ictal_idx, nonictal_idx]))

    return (
        X[keep_idx],
        y_eval[keep_idx],
        y_train[keep_idx],
        mask[keep_idx],
        t_bounds[keep_idx],
        run_ids[keep_idx],
    )



In [5]:
# ============================================================
# DATASET
# ============================================================

class EEGDataset(Dataset):
    def __init__(self, X: np.ndarray, y_train: np.ndarray, mask: np.ndarray):
        keep = y_train != -1
        self.X = torch.from_numpy(X[keep]).float()
        self.y = torch.from_numpy(y_train[keep]).float()
        self.mask = torch.from_numpy(mask[keep]).bool()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]


class EEGEvalDataset(Dataset):
    def __init__(self, X: np.ndarray, y_eval: np.ndarray, mask: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y_eval).float()
        self.mask = torch.from_numpy(mask).bool()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]


In [6]:
# ============================================================
# MODEL
# ============================================================

class ConvFeatureExtractor(nn.Module):
    def __init__(self, in_time: int = 384, cnn_hidden: int = 32, embed_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, cnn_hidden, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(cnn_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(cnn_hidden, cnn_hidden * 2, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(cnn_hidden * 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Conv1d(cnn_hidden * 2, embed_dim, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        z = self.conv(x)
        z = self.pool(z).squeeze(-1)
        return z


class SeizureCNNTransformer(nn.Module):
    def __init__(
        self,
        embed_dim: int = 64,
        nhead: int = 2,
        num_layers: int = 1,
        ff_mult: int = 4,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.feature = ConvFeatureExtractor(
            in_time=384,
            cnn_hidden=CNN_HIDDEN,
            embed_dim=embed_dim,
            dropout=dropout,
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=nhead,
            dim_feedforward=embed_dim * ff_mult,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.cls = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 1),
        )

    def forward(self, x, ch_mask):
        B, C, T = x.shape
        z = x.reshape(B * C, 1, T)
        z = self.feature(z)
        z = z.reshape(B, C, -1)

        key_padding_mask = ~ch_mask
        z = self.transformer(z, src_key_padding_mask=key_padding_mask)

        mask_f = ch_mask.float().unsqueeze(-1)
        z_sum = (z * mask_f).sum(dim=1)
        denom = mask_f.sum(dim=1).clamp(min=1.0)
        z_mean = z_sum / denom

        logits = self.cls(z_mean).squeeze(-1)
        return logits


In [7]:
# ============================================================
# LOSS
# ============================================================

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.5, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_t * ((1 - pt) ** self.gamma) * bce
        return loss.mean()

In [8]:
# ============================================================
# TRAIN / EVAL
# ============================================================

@torch.no_grad()
def predict_probs(model: nn.Module, loader: DataLoader, device: str) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_probs, all_y = [], []
    for X, mask, y in loader:
        X = X.to(device)
        mask = mask.to(device)
        logits = model(X, mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_y.append(y.numpy())
    return np.concatenate(all_probs), np.concatenate(all_y)


def train_one_fold(
    train_data: Dict[str, np.ndarray],
    val_data: Dict[str, np.ndarray],
    test_data: Dict[str, np.ndarray],
    device: str = DEVICE,
):
    Xtr, ytr_eval, ytr_train, mtr, tbtr, ridtr = retain_subject_quota(
        train_data["X"], train_data["y"], train_data["y_train"],
        train_data["mask"], train_data["t_bounds"], train_data["run_ids"],
        max_ictal=MAX_TRAIN_ICTAL_PER_SUBJECT,
        max_nonictal=MAX_TRAIN_NONICTAL_PER_SUBJECT,
        seed=SEED,
    )

    train_ds = EEGDataset(Xtr, ytr_train, mtr)
    val_ds = EEGEvalDataset(val_data["X"], val_data["y"], val_data["mask"])
    test_ds = EEGEvalDataset(test_data["X"], test_data["y"], test_data["mask"])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = SeizureCNNTransformer(
        embed_dim=EMBED_DIM,
        nhead=NHEAD,
        num_layers=NUM_LAYERS,
        ff_mult=FF_MULT,
        dropout=DROPOUT,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = BinaryFocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)

    best_state = None
    best_score = -1.0
    patience = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for X, mask, y in train_loader:
            X = X.to(device)
            mask = mask.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(X, mask)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(y)

        train_loss = running_loss / max(1, len(train_ds))

        val_prob, val_y = predict_probs(model, val_loader, device)
        val_prob_s = moving_average(val_prob, SMOOTHING_KERNEL)

        val_thr = select_best_threshold(
            val_y.astype(np.int64),
            val_prob_s.astype(np.float32),
            THRESH_GRID,
            fallback_threshold=FALLBACK_THRESHOLD,
        )
        val_metrics = compute_metrics(val_y.astype(np.int64), val_prob_s, val_thr)

        score = val_metrics["balanced_acc"]
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_threshold": val_thr,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOPPING_PATIENCE:
                break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    val_prob, val_y = predict_probs(model, val_loader, device)
    val_prob_s = moving_average(val_prob, SMOOTHING_KERNEL)
    best_thr = select_best_threshold(
        val_y.astype(np.int64),
        val_prob_s.astype(np.float32),
        THRESH_GRID,
        fallback_threshold=FALLBACK_THRESHOLD,
    )

    test_prob, test_y = predict_probs(model, test_loader, device)
    test_prob_s = moving_average(test_prob, SMOOTHING_KERNEL)
    test_metrics = compute_metrics(test_y.astype(np.int64), test_prob_s, best_thr)

    return {
        "model_state": best_state,
        "history": history,
        "best_threshold": best_thr,
        "val_prob": val_prob_s,
        "val_y": val_y,
        "test_prob": test_prob_s,
        "test_y": test_y,
        "test_metrics": test_metrics,
        "train_n_after_retention": len(train_ds),
    }

In [9]:
# ============================================================
# HOLDOUT SELECTION
# ============================================================

def choose_llm_holdout_runs(subject_name: str, records: List[RunRecord]) -> Dict[str, str]:
    if subject_name in MANUAL_LLM_HOLDOUTS:
        chosen = MANUAL_LLM_HOLDOUTS[subject_name]
        if "ictal" not in chosen or "interictal" not in chosen:
            raise RuntimeError(
                f"{subject_name}: MANUAL_LLM_HOLDOUTS must contain both 'ictal' and 'interictal'."
            )
        return chosen

    ictal_runs = sorted(
        [r for r in records if r.task == "ictal"],
        key=lambda r: (r.n_ictal, r.run_stem),
    )
    interictal_runs = sorted(
        [r for r in records if r.task == "interictal"],
        key=lambda r: (r.n_nonictal, r.run_stem),
    )

    if len(ictal_runs) < 2:
        raise RuntimeError(f"{subject_name}: need at least 2 ictal runs for LLM holdout strategy.")
    if len(interictal_runs) < 1:
        raise RuntimeError(f"{subject_name}: need at least 1 interictal run for LLM holdout strategy.")

    return {
        "ictal": ictal_runs[-1].run_stem,
        "interictal": interictal_runs[-1].run_stem,
    }


def split_train_vs_llm_holdout(
    subject_name: str,
    records: List[RunRecord],
) -> Tuple[List[RunRecord], List[RunRecord], Dict[str, str] | None]:
    if subject_name not in LLM_COHORT_SUBJECTS:
        return records, [], None

    chosen = choose_llm_holdout_runs(subject_name, records)
    reserved_ids = {chosen["ictal"], chosen["interictal"]}

    train_records = [r for r in records if r.run_stem not in reserved_ids]
    reserved_records = [r for r in records if r.run_stem in reserved_ids]

    if len(train_records) < 2:
        raise RuntimeError(f"{subject_name}: too few training runs remain after reserving LLM holdouts.")

    return train_records, reserved_records, chosen


def choose_validation_run(train_records: List[RunRecord]) -> str:
    ictal_runs = [r for r in train_records if r.task == "ictal"]
    if not ictal_runs:
        raise RuntimeError("No ictal runs available for validation selection.")
    ictal_runs = sorted(ictal_runs, key=lambda r: (r.n_ictal, r.run_stem))
    return ictal_runs[0].run_stem


def build_subject_run_map(records: List[RunRecord]) -> Dict[str, RunRecord]:
    return {r.run_stem: r for r in records}



In [10]:
# ============================================================
# RESERVED RUN INFERENCE / JSON EXPORT
# ============================================================

def evaluate_reserved_runs(
    model: nn.Module,
    reserved_data: Dict[str, np.ndarray],
    threshold: float,
    device: str = DEVICE,
) -> Dict[str, Any]:
    eval_ds = EEGEvalDataset(
        reserved_data["X"],
        reserved_data["y"],
        reserved_data["mask"],
    )
    eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)

    prob_raw, y_true = predict_probs(model, eval_loader, device)
    prob_smooth = moving_average(prob_raw, SMOOTHING_KERNEL)
    pred = (prob_smooth >= threshold).astype(np.int64)

    metrics = compute_metrics(y_true.astype(np.int64), prob_smooth, threshold)

    return {
        "y_true": y_true.astype(np.int64),
        "prob_raw": prob_raw.astype(np.float32),
        "prob_smooth": prob_smooth.astype(np.float32),
        "pred": pred.astype(np.int64),
        "run_ids": reserved_data["run_ids"],
        "t_bounds": reserved_data["t_bounds"],
        "threshold_used": float(threshold),
        "acc": metrics["acc"],
        "balanced_acc": metrics["balanced_acc"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "auroc": metrics["auroc"],
        "auprc": metrics["auprc"],
        "n_test": int(len(y_true)),
        "n_ictal": int((y_true == 1).sum()),
        "n_nonictal": int((y_true == 0).sum()),
        "tn": metrics["tn"],
        "fp": metrics["fp"],
        "fn": metrics["fn"],
        "tp": metrics["tp"],
    }


def summarize_positive_segments(pred_df: pd.DataFrame) -> List[dict]:
    if pred_df.empty:
        return []

    pos = pred_df[pred_df["pred"] == 1].copy()
    if pos.empty:
        return []

    pos = pos.sort_values("start_sec").reset_index(drop=True)

    segments = []
    cur_start = float(pos.loc[0, "start_sec"])
    cur_end = float(pos.loc[0, "end_sec"])
    cur_probs = [float(pos.loc[0, "prob_smooth"])]
    cur_n = 1

    for i in range(1, len(pos)):
        row = pos.loc[i]
        start_i = float(row["start_sec"])
        end_i = float(row["end_sec"])
        prob_i = float(row["prob_smooth"])

        if start_i <= cur_end + 1e-6:
            cur_end = max(cur_end, end_i)
            cur_probs.append(prob_i)
            cur_n += 1
        else:
            segments.append({
                "start_sec": cur_start,
                "end_sec": cur_end,
                "duration_sec": cur_end - cur_start,
                "n_positive_windows": cur_n,
                "mean_prob_smooth": float(np.mean(cur_probs)),
                "peak_prob_smooth": float(np.max(cur_probs)),
            })
            cur_start = start_i
            cur_end = end_i
            cur_probs = [prob_i]
            cur_n = 1

    segments.append({
        "start_sec": cur_start,
        "end_sec": cur_end,
        "duration_sec": cur_end - cur_start,
        "n_positive_windows": cur_n,
        "mean_prob_smooth": float(np.mean(cur_probs)),
        "peak_prob_smooth": float(np.max(cur_probs)),
    })
    return segments


def build_run_level_prediction_df(rev: Dict[str, Any]) -> pd.DataFrame:
    df = pd.DataFrame({
        "run_id": rev["run_ids"],
        "y_true": rev["y_true"],
        "prob_raw": rev["prob_raw"],
        "prob_smooth": rev["prob_smooth"],
        "pred": rev["pred"],
        "start_sec": rev["t_bounds"][:, 0],
        "end_sec": rev["t_bounds"][:, 1],
    })
    return df.sort_values(["run_id", "start_sec", "end_sec"]).reset_index(drop=True)


def build_reserved_run_json(
    subject_id: str,
    run_id: str,
    run_task: str,
    run_df: pd.DataFrame,
    threshold_used: float,
) -> Dict[str, Any]:
    positive_segments = summarize_positive_segments(run_df)

    pred_pos = int((run_df["pred"] == 1).sum())
    true_pos = int((run_df["y_true"] == 1).sum())

    return {
        "subject_id": subject_id,
        "module": "seizure_detection",
        "dataset": "HUP",
        "run_id": run_id,
        "task": run_task,
        "window_level_summary": {
            "n_windows": int(len(run_df)),
            "n_true_ictal_windows": true_pos,
            "n_true_nonictal_windows": int((run_df["y_true"] == 0).sum()),
            "n_predicted_positive_windows": pred_pos,
            "fraction_predicted_positive": float(pred_pos / len(run_df)) if len(run_df) else 0.0,
            "mean_prob_smooth": float(run_df["prob_smooth"].mean()) if len(run_df) else 0.0,
            "peak_prob_smooth": float(run_df["prob_smooth"].max()) if len(run_df) else 0.0,
            "threshold_used": float(threshold_used),
        },
        "run_level_interpretation": {
            "predicted_ictal_present": bool(pred_pos > 0),
            "first_positive_start_sec": float(run_df.loc[run_df["pred"] == 1, "start_sec"].min()) if pred_pos > 0 else None,
            "last_positive_end_sec": float(run_df.loc[run_df["pred"] == 1, "end_sec"].max()) if pred_pos > 0 else None,
            "candidate_segments": positive_segments,
        },
        "window_predictions": [
            {
                "start_sec": float(r.start_sec),
                "end_sec": float(r.end_sec),
                "y_true": int(r.y_true),
                "prob_raw": float(r.prob_raw),
                "prob_smooth": float(r.prob_smooth),
                "pred": int(r.pred),
            }
            for r in run_df.itertuples(index=False)
        ],
        "llm_notes": [
            "Use only information explicitly present in this JSON.",
            "Do not infer channel-level seizure onset from this file because channel-wise outputs are not included.",
            "Candidate segments are derived from consecutive positive windows after smoothing and thresholding.",
            "If positive windows are diffuse or frequent in interictal runs, state uncertainty and possible false positives."
        ],
    }


def export_reserved_run_jsons(
    subject_id: str,
    chosen: Dict[str, str],
    rev: Dict[str, Any],
):
    llm_seizure_dir = LLM_UNIFIED_ROOT / subject_id / "seizure"
    llm_seizure_dir.mkdir(parents=True, exist_ok=True)

    pred_df = build_run_level_prediction_df(rev)

    run_task_map = {
        chosen["ictal"]: "ictal",
        chosen["interictal"]: "interictal",
    }

    for run_id, run_task in run_task_map.items():
        run_df = pred_df[pred_df["run_id"] == run_id].copy().reset_index(drop=True)
        if run_df.empty:
            print(f"[WARN] No reserved predictions found for {run_id}")
            continue

        run_json = build_reserved_run_json(
            subject_id=subject_id,
            run_id=run_id,
            run_task=run_task,
            run_df=run_df,
            threshold_used=rev["threshold_used"],
        )

        out_name = f"{run_id}_seizure_summary_for_llm.json"
        out_path = llm_seizure_dir / out_name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(run_json, f, indent=2)

        print(f"Saved LLM seizure JSON: {out_path}")



In [11]:
# ============================================================
# NAMING-FIX EXPORT HELPERS
# ============================================================

class EEGEvalDataset(Dataset):
    def __init__(self, X: np.ndarray, y_eval: np.ndarray, mask: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y_eval).float()
        self.mask = torch.from_numpy(mask).bool()
    def __len__(self) -> int: return len(self.y)
    def __getitem__(self, idx: int): return self.X[idx], self.mask[idx], self.y[idx]

def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1 or len(x) == 0: return x.copy()
    pad = k // 2
    xpad = np.pad(x, (pad, pad), mode='edge')
    kernel = np.ones(k, dtype=np.float32) / float(k)
    return np.convolve(xpad, kernel, mode='valid')

def predict_probs_with_logits(model: nn.Module, loader: DataLoader, device: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval(); all_probs=[]; all_logits=[]; all_y=[]
    with torch.no_grad():
        for X, mask, y in loader:
            X = X.to(device); mask = mask.to(device)
            logits = model(X, mask); probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy()); all_logits.append(logits.cpu().numpy()); all_y.append(y.numpy())
    return np.concatenate(all_probs).astype(np.float32), np.concatenate(all_logits).astype(np.float32), np.concatenate(all_y).astype(np.int64)

def evaluate_reserved_runs_named(*, model: nn.Module, reserved_data: Dict[str,np.ndarray], threshold: float, smoothing_kernel: int, batch_size: int, device: str, compute_metrics_fn):
    eval_ds = EEGEvalDataset(reserved_data['X'], reserved_data['y'], reserved_data['mask'])
    eval_loader = DataLoader(eval_ds, batch_size=batch_size, shuffle=False)
    prob_raw, logits_raw, y_true = predict_probs_with_logits(model, eval_loader, device)
    prob_smooth = moving_average(prob_raw, smoothing_kernel).astype(np.float32)
    pred = (prob_smooth >= threshold).astype(np.int64)
    metrics = compute_metrics_fn(y_true.astype(np.int64), prob_smooth, threshold)
    return {'y_true': y_true.astype(np.int64), 'prob_raw': prob_raw.astype(np.float32), 'logits_raw': logits_raw.astype(np.float32), 'prob_smooth': prob_smooth.astype(np.float32), 'pred': pred.astype(np.int64), 'run_ids': reserved_data['run_ids'], 't_bounds': reserved_data['t_bounds'], 'mask': reserved_data['mask'], 'threshold_used': float(threshold), 'acc': metrics['acc'], 'balanced_acc': metrics['balanced_acc'], 'precision': metrics['precision'], 'recall': metrics['recall'], 'f1': metrics['f1'], 'auroc': metrics['auroc'], 'auprc': metrics['auprc'], 'n_test': int(len(y_true)), 'n_ictal': int((y_true==1).sum()), 'n_nonictal': int((y_true==0).sum()), 'tn': metrics['tn'], 'fp': metrics['fp'], 'fn': metrics['fn'], 'tp': metrics['tp']}

def load_meta(meta_path: Path) -> Dict[str, Any]:
    with open(meta_path, 'r', encoding='utf-8') as f: return json.load(f)

def build_model_channel_names_from_meta(meta_path: Path) -> List[str]:
    meta = load_meta(meta_path)
    ref_names = meta.get('channel_names_after_reference', [])
    n_after_fix128 = int(meta.get('n_channels_after_fix128', len(ref_names)))
    kept_idx = meta.get('kept_channel_indices', list(range(len(ref_names))))
    if not ref_names: raise ValueError(f'Missing channel_names_after_reference in {meta_path}')
    model_names=[]
    for i in range(n_after_fix128):
        if i < len(kept_idx):
            src_idx = int(kept_idx[i])
            model_names.append(str(ref_names[src_idx]) if 0 <= src_idx < len(ref_names) else f'UNKNOWN_SRC_{src_idx}')
        else:
            model_names.append('PAD')
    return model_names

def clone_for_saliency(model: nn.Module, device: str) -> nn.Module:
    model_s = copy.deepcopy(model).to(device); model_s.eval()
    for p in model_s.parameters(): p.requires_grad_(False)
    return model_s

def compute_channel_saliency_for_window(model: nn.Module, x_win: np.ndarray, mask_win: np.ndarray, device: str) -> np.ndarray:
    x = torch.from_numpy(x_win[None]).float().to(device); x.requires_grad_(True)
    mask = torch.from_numpy(mask_win[None]).bool().to(device)
    model.zero_grad(set_to_none=True); logits = model(x, mask); logits[0].backward()
    grad = x.grad.detach().cpu().numpy()[0]; signal = x.detach().cpu().numpy()[0]
    sal = np.mean(np.abs(grad * signal), axis=1); sal = sal * mask_win.astype(np.float32)
    denom = float(sal.sum());
    if denom > 0: sal = sal / denom
    return sal.astype(np.float32)

def compute_run_channel_saliency_named(*, model: nn.Module, run_data: Dict[str,np.ndarray], model_channel_names: Sequence[str], device: str, positive_only: bool = True, top_k: int = 10) -> Dict[str, Any]:
    model_s = clone_for_saliency(model, device)
    X=run_data['X']; mask=run_data['mask']; prob_smooth=run_data['prob_smooth']; pred=run_data['pred']; t_bounds=run_data['t_bounds']; y_true=run_data['y_true']
    idx = np.where(pred==1)[0] if positive_only else np.arange(len(X))
    if len(idx)==0: idx=np.array([int(np.argmax(prob_smooth))])
    per_window_rows=[]; sal_list=[]
    for wi in idx.tolist():
        sal = compute_channel_saliency_for_window(model_s, X[wi], mask[wi], device); sal_list.append(sal)
        ranked_idx = np.argsort(sal)[::-1]
        top_channels=[]
        for j in ranked_idx[:top_k]:
            if not bool(mask[wi,j]): continue
            ch_name = str(model_channel_names[j])
            if ch_name == 'PAD': continue
            top_channels.append({'model_channel': f'CH{j+1:03d}', 'channel': ch_name, 'saliency': float(sal[j])})
        per_window_rows.append({'window_index': int(wi), 'start_sec': float(t_bounds[wi,0]), 'end_sec': float(t_bounds[wi,1]), 'y_true': int(y_true[wi]), 'prob_smooth': float(prob_smooth[wi]), 'pred': int(pred[wi]), 'top_channels': top_channels})
    sal_mean = np.mean(np.stack(sal_list, axis=0), axis=0) if sal_list else np.zeros(len(model_channel_names), dtype=np.float32)
    ranked_idx = np.argsort(sal_mean)[::-1]
    aggregated=[]
    for j in ranked_idx:
        ch_name = str(model_channel_names[j])
        if ch_name == 'PAD' or sal_mean[j] <= 0: continue
        aggregated.append({'model_channel': f'CH{j+1:03d}', 'channel': ch_name, 'mean_saliency': float(sal_mean[j]), 'supporting_positive_windows': int(sum(1 for row in per_window_rows if any(tc['channel']==ch_name for tc in row['top_channels'])))})
    early_rows = sorted(per_window_rows, key=lambda r:r['start_sec'])[:min(3, len(per_window_rows))]
    counter={}
    for row in early_rows:
        for tc in row['top_channels'][:min(5, len(row['top_channels']))]:
            key=(tc['model_channel'], tc['channel']); counter[key]=counter.get(key,0)+1
    early_ictal_channels_topk=[{'model_channel':mc,'channel':ch,'early_window_hits':hits} for (mc,ch),hits in sorted(counter.items(), key=lambda kv:(-kv[1], kv[0][0]))[:top_k]]
    return {'method':'gradient_x_input_saliency_proxy','positive_only':bool(positive_only),'n_windows_used':int(len(idx)),'channel_name_mapping':{f'CH{i+1:03d}':str(ch) for i,ch in enumerate(model_channel_names)},'channel_onset_scores':aggregated,'early_ictal_channels_topk':early_ictal_channels_topk,'window_level_channel_evidence':per_window_rows,'limitations':['Channel evidence is a model-saliency proxy, not direct electrophysiological onset annotation.','Scores depend on the trained seizure classifier and current preprocessing/windowing choices.','Channels labeled PAD are model padding positions and are excluded from interpretation.']}

def build_run_level_prediction_df(rev: Dict[str, Any]) -> pd.DataFrame:
    return pd.DataFrame({'run_id': rev['run_ids'],'y_true': rev['y_true'],'prob_raw': rev['prob_raw'],'logit': rev['logits_raw'],'prob_smooth': rev['prob_smooth'],'pred': rev['pred'],'start_sec': rev['t_bounds'][:,0],'end_sec': rev['t_bounds'][:,1]}).sort_values(['run_id','start_sec','end_sec']).reset_index(drop=True)

def summarize_positive_segments(pred_df: pd.DataFrame) -> List[dict]:
    if pred_df.empty: return []
    pos = pred_df[pred_df['pred']==1].copy()
    if pos.empty: return []
    pos = pos.sort_values('start_sec').reset_index(drop=True)
    segments=[]; cur_start=float(pos.loc[0,'start_sec']); cur_end=float(pos.loc[0,'end_sec']); cur_probs=[float(pos.loc[0,'prob_smooth'])]; cur_n=1
    for i in range(1,len(pos)):
        row=pos.loc[i]; start_i=float(row['start_sec']); end_i=float(row['end_sec']); prob_i=float(row['prob_smooth'])
        if start_i <= cur_end + 1e-6:
            cur_end=max(cur_end,end_i); cur_probs.append(prob_i); cur_n+=1
        else:
            segments.append({'start_sec':cur_start,'end_sec':cur_end,'duration_sec':cur_end-cur_start,'n_positive_windows':cur_n,'mean_prob_smooth':float(np.mean(cur_probs)),'peak_prob_smooth':float(np.max(cur_probs))})
            cur_start=start_i; cur_end=end_i; cur_probs=[prob_i]; cur_n=1
    segments.append({'start_sec':cur_start,'end_sec':cur_end,'duration_sec':cur_end-cur_start,'n_positive_windows':cur_n,'mean_prob_smooth':float(np.mean(cur_probs)),'peak_prob_smooth':float(np.max(cur_probs))})
    return segments

def build_reserved_run_json_named(*, subject_id: str, run_id: str, run_task: str, run_df: pd.DataFrame, threshold_used: float, channel_level_summary: Optional[Dict[str,Any]]) -> Dict[str, Any]:
    positive_segments = summarize_positive_segments(run_df); pred_pos=int((run_df['pred']==1).sum()); true_pos=int((run_df['y_true']==1).sum())
    return {'subject_id':subject_id,'module':'seizure_detection','dataset':'HUP','run_id':run_id,'task':run_task,'window_level_summary':{'n_windows':int(len(run_df)),'n_true_ictal_windows':true_pos,'n_true_nonictal_windows':int((run_df['y_true']==0).sum()),'n_predicted_positive_windows':pred_pos,'fraction_predicted_positive':float(pred_pos/len(run_df)) if len(run_df) else 0.0,'mean_prob_smooth':float(run_df['prob_smooth'].mean()) if len(run_df) else 0.0,'peak_prob_smooth':float(run_df['prob_smooth'].max()) if len(run_df) else 0.0,'threshold_used':float(threshold_used)},'run_level_interpretation':{'predicted_ictal_present':bool(pred_pos>0),'first_positive_start_sec':float(run_df.loc[run_df['pred']==1,'start_sec'].min()) if pred_pos>0 else None,'last_positive_end_sec':float(run_df.loc[run_df['pred']==1,'end_sec'].max()) if pred_pos>0 else None,'candidate_segments':positive_segments},'channel_level_summary':channel_level_summary,'window_predictions':[{'start_sec':float(r.start_sec),'end_sec':float(r.end_sec),'y_true':int(r.y_true),'prob_raw':float(r.prob_raw),'prob_smooth':float(r.prob_smooth),'pred':int(r.pred)} for r in run_df.itertuples(index=False)],'llm_notes':['Use only information explicitly present in this JSON.','Candidate segments are derived from consecutive positive windows after smoothing and thresholding.','Channel-level evidence is reported with real bipolar channel names derived from preprocessing metadata.','If channel is PAD, it is a padded model position and must not be interpreted clinically.','Channel-level evidence remains a model-saliency proxy, not direct electrophysiological onset annotation.']}

def export_reserved_run_jsons_named(*, model: nn.Module, subject_id: str, chosen: Dict[str,str], rev: Dict[str,Any], reserved_data: Dict[str,np.ndarray], reserved_records: Sequence[Any], llm_unified_root_base: Path, device: str, top_k_channels: int = 10):
    llm_subject_root = llm_unified_root_base / f"LLM_unified_HUP_{subject_id.replace('sub-', '')}"
    llm_seizure_dir = llm_subject_root / "seizure"
    llm_seizure_dir.mkdir(parents=True, exist_ok=True)
    pred_df = build_run_level_prediction_df(rev); record_map = {r.run_stem: r for r in reserved_records}
    for run_id, run_task in {chosen['ictal']:'ictal', chosen['interictal']:'interictal'}.items():
        run_df = pred_df[pred_df['run_id']==run_id].copy().reset_index(drop=True)
        if run_df.empty:
            print(f'[WARN] No reserved predictions found for {run_id}'); continue
        idx = np.where(rev['run_ids']==run_id)[0]
        run_arrays = {'X': reserved_data['X'][idx], 'y_true': rev['y_true'][idx], 'mask': reserved_data['mask'][idx], 'prob_smooth': rev['prob_smooth'][idx], 'pred': rev['pred'][idx], 't_bounds': reserved_data['t_bounds'][idx]}
        record = record_map[run_id]; model_channel_names = build_model_channel_names_from_meta(record.meta_path)
        channel_level_summary = compute_run_channel_saliency_named(model=model, run_data=run_arrays, model_channel_names=model_channel_names, device=device, positive_only=True, top_k=top_k_channels)
        run_json = build_reserved_run_json_named(subject_id=subject_id, run_id=run_id, run_task=run_task, run_df=run_df, threshold_used=rev['threshold_used'], channel_level_summary=channel_level_summary)
        out_path = llm_seizure_dir / f'{run_id}_seizure_summary_for_llm.json'
        with open(out_path,'w',encoding='utf-8') as f: json.dump(run_json,f,indent=2)
        print(f'Saved named seizure JSON: {out_path}')


In [12]:
# ============================================================
# MAIN
# ============================================================

def group_runs_by_subject(runs: List[RunRecord]) -> Dict[str, List[RunRecord]]:
    grouped: Dict[str, List[RunRecord]] = {}
    for r in runs:
        grouped.setdefault(r.subject_id, []).append(r)
    return grouped


def main():
    EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

    all_runs = scan_runs(PREPROCESSED_ROOT)
    all_runs = filter_runs(all_runs)

    if not all_runs:
        raise RuntimeError(f"No runs found in {PREPROCESSED_ROOT}")

    subject_to_runs = group_runs_by_subject(all_runs)
    subject_names = sorted(subject_to_runs.keys())

    all_internal_rows = []
    all_reserved_rows = []
    cohort_manifest = []

    for subject_name in subject_names:
        print("\n" + "#" * 120)
        print(f"SUBJECT: {subject_name}")
        print("#" * 120)

        records = sorted(subject_to_runs[subject_name], key=lambda r: r.run_stem)
        if len(records) < 2:
            print(f"Skipping {subject_name}: fewer than 2 runs.")
            continue

        train_records, reserved_records, chosen = split_train_vs_llm_holdout(subject_name, records)

        out_dir = EXPERIMENT_ROOT / subject_name
        out_dir.mkdir(parents=True, exist_ok=True)

        if chosen is not None:
            cohort_manifest.append({
                "subject_id": subject_name,
                "reserved_ictal_run": chosen["ictal"],
                "reserved_interictal_run": chosen["interictal"],
                "n_train_runs_remaining": len(train_records),
                "n_reserved_runs": len(reserved_records),
            })
            with open(out_dir / "llm_reserved_runs.json", "w", encoding="utf-8") as f:
                json.dump(chosen, f, indent=2)

        run_map = build_subject_run_map(train_records)
        ictal_train_records = [r for r in train_records if r.task == "ictal"]

        if len(ictal_train_records) < 2:
            print(f"Skipping {subject_name}: need at least 2 ictal training runs for internal CV.")
            continue

        internal_rows = []

        for fold_id, heldout in enumerate(sorted(ictal_train_records, key=lambda r: r.run_stem)):
            remaining = [r for r in train_records if r.run_stem != heldout.run_stem]
            val_run_id = choose_validation_run(remaining)

            val_rec = run_map[val_run_id]
            train_fold_records = [r for r in remaining if r.run_stem != val_run_id]

            train_paths = [r.npz_path for r in train_fold_records]
            val_paths = [val_rec.npz_path]
            test_paths = [heldout.npz_path]

            train_data = concatenate_runs(train_paths)
            val_data = concatenate_runs(val_paths)
            test_data = concatenate_runs(test_paths)

            print("=" * 100)
            print(f"{subject_name} | INTERNAL FOLD {fold_id:02d} | TEST={heldout.run_stem} | VAL={val_run_id}")
            print("=" * 100)

            res = train_one_fold(train_data, val_data, test_data, device=DEVICE)

            row = {
                "subject_id": subject_name,
                "fold_id": fold_id,
                "held_out_run": heldout.run_stem,
                "val_run": val_run_id,
                "best_threshold": res["best_threshold"],
                "train_n_after_retention": res["train_n_after_retention"],
                **{f"test_{k}": v for k, v in res["test_metrics"].items()},
            }
            internal_rows.append(row)

            fold_dir = out_dir / f"internal_fold_{fold_id:02d}"
            fold_dir.mkdir(parents=True, exist_ok=True)

            pd.DataFrame(res["history"]).to_csv(fold_dir / "training_history.csv", index=False)
            pd.DataFrame({
                "run_id": test_data["run_ids"],
                "y_true": res["test_y"],
                "prob_smooth": res["test_prob"],
                "pred": (res["test_prob"] >= res["best_threshold"]).astype(np.int64),
                "start_sec": test_data["t_bounds"][:, 0],
                "end_sec": test_data["t_bounds"][:, 1],
            }).to_csv(fold_dir / "test_predictions.csv", index=False)

        df_internal = pd.DataFrame(internal_rows).sort_values("fold_id")
        df_internal.to_csv(out_dir / "internal_cv_results.csv", index=False)
        all_internal_rows.append(df_internal)

        if reserved_records:
            # rebuild run_map from train_records, because that is the set used for final-model fitting
            run_map_final = build_subject_run_map(train_records)

            val_run_id = choose_validation_run(train_records)
            val_rec = run_map_final[val_run_id]
            final_train_records = [r for r in train_records if r.run_stem != val_run_id]

            train_paths = [r.npz_path for r in final_train_records]
            val_paths = [val_rec.npz_path]
            reserved_paths = [r.npz_path for r in reserved_records]

            train_data = concatenate_runs(train_paths)
            val_data = concatenate_runs(val_paths)
            reserved_data = concatenate_runs(reserved_paths)

            final_res = train_one_fold(train_data, val_data, reserved_data, device=DEVICE)

            pd.DataFrame(final_res["history"]).to_csv(out_dir / "final_training_history.csv", index=False)

            final_model = SeizureCNNTransformer(
                embed_dim=EMBED_DIM,
                nhead=NHEAD,
                num_layers=NUM_LAYERS,
                ff_mult=FF_MULT,
                dropout=DROPOUT,
            ).to(DEVICE)
            final_model.load_state_dict(final_res["model_state"])
            torch.save(final_res["model_state"], out_dir / "final_model.pt")

            rev = evaluate_reserved_runs_named(
                model=final_model,
                reserved_data=reserved_data,
                threshold=final_res["best_threshold"],
                smoothing_kernel=SMOOTHING_KERNEL,
                batch_size=BATCH_SIZE,
                device=DEVICE,
                compute_metrics_fn=compute_metrics,
            )

            reserved_pred_df = pd.DataFrame({
                "run_id": rev["run_ids"],
                "start_sec": rev["t_bounds"][:, 0],
                "end_sec": rev["t_bounds"][:, 1],
                "y_true": rev["y_true"],
                "prob_raw": rev["prob_raw"],
                "prob_smooth": rev["prob_smooth"],
                "pred": rev["pred"],
            }).sort_values(["run_id", "start_sec"]).reset_index(drop=True)

            reserved_pred_df.to_csv(out_dir / "llm_reserved_predictions.csv", index=False)

            export_reserved_run_jsons_named(
                model=final_model,
                subject_id=subject_name,
                chosen=chosen,
                rev=rev,
                reserved_data=reserved_data,
                reserved_records=reserved_records,
                llm_unified_root_base=LLM_UNIFIED_ROOT_BASE,
                device=DEVICE,
                top_k_channels=10,
            )

            all_reserved_rows.append({
                "subject_id": subject_name,
                "reserved_ictal_run": chosen["ictal"],
                "reserved_interictal_run": chosen["interictal"],
                "reserved_threshold_used": rev["threshold_used"],
                "reserved_acc": rev["acc"],
                "reserved_balanced_acc": rev["balanced_acc"],
                "reserved_precision": rev["precision"],
                "reserved_recall": rev["recall"],
                "reserved_f1": rev["f1"],
                "reserved_auroc": rev["auroc"],
                "reserved_auprc": rev["auprc"],
                "reserved_n_test": rev["n_test"],
                "reserved_n_ictal": rev["n_ictal"],
                "reserved_n_nonictal": rev["n_nonictal"],
                "tn": rev["tn"],
                "fp": rev["fp"],
                "fn": rev["fn"],
                "tp": rev["tp"],
            })

    if cohort_manifest:
        pd.DataFrame(cohort_manifest).to_csv(EXPERIMENT_ROOT / "llm_cohort_manifest.csv", index=False)
    if all_internal_rows:
        pd.concat(all_internal_rows, ignore_index=True).to_csv(EXPERIMENT_ROOT / "all_internal_cv_results.csv", index=False)
    if all_reserved_rows:
        pd.DataFrame(all_reserved_rows).to_csv(EXPERIMENT_ROOT / "all_llm_reserved_results.csv", index=False)

    print("\nSaved outputs to:", EXPERIMENT_ROOT)


if __name__ == "__main__":
    main()





########################################################################################################################
SUBJECT: sub-HUP126
########################################################################################################################
sub-HUP126 | INTERNAL FOLD 00 | TEST=sub-HUP126_ses-presurgery_task-ictal_acq-ecog_run-01 | VAL=sub-HUP126_ses-presurgery_task-ictal_acq-ecog_run-03


C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


sub-HUP126 | INTERNAL FOLD 01 | TEST=sub-HUP126_ses-presurgery_task-ictal_acq-ecog_run-02 | VAL=sub-HUP126_ses-presurgery_task-ictal_acq-ecog_run-01


C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


sub-HUP126 | INTERNAL FOLD 02 | TEST=sub-HUP126_ses-presurgery_task-ictal_acq-ecog_run-03 | VAL=sub-HUP126_ses-presurgery_task-ictal_acq-ecog_run-01


...

Saved named seizure JSON: D:\LLM_outputs_for_subjects\LLM_unified_HUP_HUP157\seizure\sub-HUP157_ses-presurgery_task-ictal_acq-seeg_run-03_seizure_summary_for_llm.json
Saved named seizure JSON: D:\LLM_outputs_for_subjects\LLM_unified_HUP_HUP157\seizure\sub-HUP157_ses-presurgery_task-interictal_acq-seeg_run-02_seizure_summary_for_llm.json

########################################################################################################################
SUBJECT: sub-HUP164
########################################################################################################################
sub-HUP164 | INTERNAL FOLD 00 | TEST=sub-HUP164_ses-presurgery_task-ictal_acq-seeg_run-01 | VAL=sub-HUP164_ses-presurgery_task-ictal_acq-seeg_run-02


C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


sub-HUP164 | INTERNAL FOLD 01 | TEST=sub-HUP164_ses-presurgery_task-ictal_acq-seeg_run-02 | VAL=sub-HUP164_ses-presurgery_task-ictal_acq-seeg_run-01


C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
C:\Users\ajars\anaconda3\envs\braindecode_env\lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Saved named seizure JSON: D:\LLM_outputs_for_subjects\LLM_unified_HUP_HUP164\seizure\sub-HUP164_ses-presurgery_task-ictal_acq-seeg_run-03_seizure_summary_for_llm.json
Saved named seizure JSON: D:\LLM_outputs_for_subjects\LLM_unified_HUP_HUP164\seizure\sub-HUP164_ses-presurgery_task-interictal_acq-seeg_run-02_seizure_summary_for_llm.json

Saved outputs to: D:\hup_all_subjects_ver2
